Bohua Liu is responsible for this code

Oversample Rare Data

In [2]:
from pathlib import Path
from collections import defaultdict

# === Settings ===
img_dir = Path("dataset_yolo_4cats_seg/images/train")
labels_dir = Path("dataset_yolo_4cats_seg/labels/train")
output_txt = img_dir.parent / "train_oversampled.txt"

names = ["plastic", "glass", "paper", "unsorted"]
class_weights = {
    "plastic": 1,
    "glass": 5,
    "paper": 3,
    "unsorted": 1
}
idx_to_class = {i: name for i, name in enumerate(names)}

# === Build image list ===
image_paths = sorted(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")))
oversampled_lines = []

for img_path in image_paths:
    label_path = labels_dir / (img_path.stem + ".txt")
    if not label_path.exists():
        continue

    with open(label_path, "r") as f:
        lines = f.read().splitlines()
    class_ids = {int(l.split()[0]) for l in lines if l.strip()}

    if not class_ids:
        repeat_factor = 1
    else:
        classes = [idx_to_class[c] for c in class_ids]
        repeat_factor = max(class_weights.get(c, 1) for c in classes)

    oversampled_lines.extend([str(img_path.resolve())] * repeat_factor)

# === Write to file ===
with open(output_txt, "w") as f:
    f.write("\n".join(oversampled_lines))

print(f"Oversampled train list saved to: {output_txt}")


Oversampled train list saved to: dataset_yolo_4cats_seg/images/train_oversampled.txt


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
from ultralytics import YOLO

yaml_directory = "dataset_yolo_4cats_seg/taco_oversampled.yaml"

# 1. Initialize the P2 architecture (Small)
# The filename "yolov8s-p2.yaml" tells YOLO to use the 's' scale from the file.
model = YOLO('yolov8s-p2.yaml')

# 2. Transfer weights from standard YOLOv8s
# This is crucial. If you don't do this, you start with random weights (0% accuracy).
# You will get a warning about shape mismatch (because of the extra P2 layer) - Ignore it.
model = model.load('yolov8s.pt') 

results = model.train(
   data=yaml_directory,
    
    # SYSTEM SETTINGS
    epochs=300,               # Increased: 100 is often slightly too short for convergence on complex data.
    # patience=30,              # Early Stopping: If no improvement for 30 epochs, stop. Saves time.
    batch=8,                 # Standard: Try 16. If you get "CUDA Out of Memory", drop to 8 or 4.
    name="yolov8s_p2_oversampled",
    project="Yolo_traning",
    seed=42,
    
    # IMAGE SETTINGS (Crucial for TACO)
    imgsz=1024,               # HIGH PRIORITY: TACO images are high res. 
                              # Downscaling to 640 makes cigarettes vanish. 
                              # If 1024 crashes your GPU, try 800.
    
    # AUGMENTATION (Trash specific tweaks)
    # Trash has no specific orientation (unlike a car or a person).
    degrees=45.0,             # Rotate images +/- 45 degrees (default is 0.0).
    flipud=0.5,               # Vertical Flip: 50% chance. Trash can be upside down.
    fliplr=0.5,               # Horizontal Flip: 50% chance (Standard).
    mosaic=1.0,               # Keep Mosaic on (1.0). Great for learning trash in different contexts.
    mixup=0.1,                # MixUp augmentation (default is 0.0, off). Helps with small objects.
    copy_paste=0.5, 

    # OPTIMIZATION
    optimizer='auto',         # YOLOv8 handles this well, but 'AdamW' is often great for custom datasets.
    cos_lr=True,          

    # NEW
    weight_decay=5e-4,      # Regularization

    # NEW COLOR AUGMENTATIONS
    hsv_h=0.015,     # Hue augmentation
    hsv_s=0.7,       # Strong saturation augmentation
    hsv_v=0.4,       # Strong brightness augmentation
)

Transferred 219/485 items from pretrained weights
New https://pypi.org/project/ultralytics/8.3.232 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.0 🚀 Python-3.10.12 torch-2.9.1+cu128 CUDA:0 (Tesla T4, 15948MiB)
engine/trainer: task=detect, mode=train, model=yolov8s-p2.yaml, data=dataset_yolo_4cats_seg/taco_oversampled.yaml, epochs=300, time=None, patience=100, batch=8, imgsz=1024, save=True, save_period=-1, cache=False, device=None, workers=8, project=Yolo_traning, name=yolov8s_p2_oversampled, exist_ok=False, pretrained=yolov8s.pt, optimizer=auto, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, 

train: Scanning /home/default/Desktop/project/Dataset 4cats/dataset_yolo_4cats_seg/labels/train... 2157 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2157/2157 [00:05<00:00, 399.53it/s]


train: New cache created: /home/default/Desktop/project/Dataset 4cats/dataset_yolo_4cats_seg/labels/train.cache


val: Scanning /home/default/Desktop/project/Dataset 4cats/dataset_yolo_4cats_seg/labels/val... 225 images, 0 backgrounds, 0 corrupt: 100%|██████████| 225/225 [00:01<00:00, 153.37it/s]

val: New cache created: /home/default/Desktop/project/Dataset 4cats/dataset_yolo_4cats_seg/labels/val.cache


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    138/300      14.2G     0.8319     0.8568      1.107         42       1024: 100%|██████████| 270/270 [02:01<00:00,  2.22it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 15/15 [00:03<00:00,  3.81it/s]

                   all        225        727      0.468      0.325      0.337      0.273



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    139/300      12.5G     0.8568     0.8541      1.106         91       1024:  11%|█         | 29/270 [00:13<01:52,  2.14it/s]

In [ ]:
from ultralytics import YOLO
import os
os.environ["WANDB_DISABLED"] = "true"

# Load the last saved model
model = YOLO("Yolo_traning/yolov8s_p2_oversampled/weights/last.pt")

# Resume training
model.train(
    resume=True,
    patience=30
)


/home/default/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


New https://pypi.org/project/ultralytics/8.3.232 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.0 🚀 Python-3.10.12 torch-2.9.1+cu128 CUDA:0 (Tesla T4, 15948MiB)
engine/trainer: task=detect, mode=train, model=Yolo_traning/yolov8s_p2_oversampled/weights/last.pt, data=dataset_yolo_4cats_seg/taco_oversampled.yaml, epochs=300, time=None, patience=100, batch=8, imgsz=1024, save=True, save_period=-1, cache=False, device=None, workers=8, project=Yolo_traning, name=yolov8s_p2_oversampled, exist_ok=False, pretrained=yolov8s.pt, optimizer=auto, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=Yolo_traning/yolov8s_p2_oversampled/weights/last.pt, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream

train: Scanning /home/default/Desktop/project/Dataset 4cats/dataset_yolo_4cats_seg/labels/train.cache... 2157 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2157/2157 [00:00<?, ?it/s]
val: Scanning /home/default/Desktop/project/Dataset 4cats/dataset_yolo_4cats_seg/labels/val.cache... 225 images, 0 backgrounds, 0 corrupt: 100%|██████████| 225/225 [00:00<?, ?it/s]


Plotting labels to Yolo_traning/yolov8s_p2_oversampled/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 78 weight(decay=0.0), 87 weight(decay=0.0005), 86 bias(decay=0.0)
Resuming training Yolo_traning/yolov8s_p2_oversampled/weights/last.pt from epoch 177 to 300 total epochs
Image sizes 1024 train, 1024 val
Using 8 dataloader workers
Logging results to Yolo_traning/yolov8s_p2_oversampled
Starting training for 300 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    177/300      13.4G     0.7654     0.7418      1.064         51       1024: 100%|██████████| 270/270 [02:17<00:00,  1.97it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 15/15 [00:21<00:00,  1.44s/it]

...

                   all        225        727      0.574       0.27      0.299      0.239



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    185/300      13.5G     0.8348     0.7814      1.083        123       1024:  13%|█▎        | 36/270 [00:16<01:47,  2.17it/s]


KeyboardInterrupt: 

: 

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
from ultralytics import YOLO

# Path to your dataset
yaml_directory = "dataset_yolo_4cats_seg/taco_oversampled.yaml"
model = YOLO("Yolo_traning/yolov8s_p2_oversampled/weights/best.pt")
metrics = model.val(data=yaml_directory,name="yolov8s_p2_oversampled",project="Yolo_validation")
print(metrics)

/home/default/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Ultralytics 8.3.0 🚀 Python-3.10.12 torch-2.9.1+cu128 CUDA:0 (Tesla T4, 15948MiB)
YOLOv8s-p2 summary (fused): 231 layers, 10,017,056 parameters, 0 gradients, 31.4 GFLOPs


val: Scanning /home/default/Desktop/project/Dataset 4cats/dataset_yolo_4cats_seg/labels/val.cache... 225 images, 0 backgrounds, 0 corrupt: 100%|██████████| 225/225 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 15/15 [00:06<00:00,  2.40it/s]


                   all        225        727      0.444      0.326      0.338      0.273
               plastic        168        322      0.469      0.512      0.495      0.393
                 glass         15         27      0.278      0.185      0.167      0.144
                 paper         47         71      0.481      0.324      0.336      0.303
              unsorted        110        307      0.548      0.284      0.352      0.253
Speed: 0.7ms preprocess, 20.4ms inference, 0.0ms loss, 2.2ms postprocess per image
Results saved to Yolo_validation/yolov8s_p2_oversampled
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7901fcbb3c40>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.0

: 